In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

from deepagents import create_deep_agent
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

True

# StateBackend

IT is the Langgraph State. Which intends to store the files in the RAM of the local system

In [2]:
from deepagents.backends import StateBackend

In [4]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    api_key=os.getenv("GOOGLE_API_KEY"),
)

agent = create_deep_agent(
    model=llm,
    backend=StateBackend()
)

In [5]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content: \n"
            "1. Record Video\n2. Edit Video\n3. Upload Video\n"
            "Then tell me you've done it"
        )
    }]
})

print("\n------- Agent Reply -----------------------------------------------------------")
print(result["messages"][-1].content)


------- Agent Reply -----------------------------------------------------------
I've created the file `/notes/todo.txt` with the requested content.


In [6]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content: \n1. Record Video\n2. Edit Video\n3. Upload Video\nThen tell me you've done it", additional_kwargs={}, response_metadata={}, id='232ebc60-cc6d-49eb-8b65-ce8badbb9493'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'write_file', 'arguments': '{"file_path": "/notes/todo.txt", "content": "1. Record Video\\n2. Edit Video\\n3. Upload Video"}'}, '__gemini_function_call_thought_signatures__': {'aeadcb3b-7563-4cc9-8356-38d403dae092': 'CqwCAWkUfRNO2UxzBDQPDBZphPJRCl0ay3uMjdSjgTB8SxoFDI9iCF8+HSbme1B+6Cpjhd+js+xa3PGFQKxM9n9tg9K3c29bTropsJPyDQdv174jcyZLGeDbzS2p80MWTl77d+QJVblznOUOH3oAc3jgZI3nfVY357faAzqbYa3+dvib435hM5N6dgqCbKT1yb0Etr0g+BfZ9OCaF2E4Fx9W5cgo4MpF0a8W4vodwimC00NjzPOyJip92XAL4UOwMgvxNsZ6tahElVPtwxMaha9WPVylD9007m8b7Pdq9i8S7vH3DCcd/8bbnij9s6Ezw1hF5otu2JpMMq2EFtwf6+D69WFIbkCKKglG9Ir7qcYbbttnhdgbcLkIovoEJKkUwEXHYoUnb2pRxBysyUc1'}}, response_metadata={'finish_reason': 'STO

In [8]:
# Check the Backend is working with StateBackend, written files appear under result["files"]

print("\n-------- Backend Check -------------------------------------------------------------------------------------------------------------")
files = result.get("files", {})

if files:
    print(f" State Backend is working - {len(files)} files in state:")
    for path, content in files.items():
        print(f"\n {path}\n{'-' * 40}\n{content}")
else:
    print("No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly"
          )


-------- Backend Check -------------------------------------------------------------------------------------------------------------
 State Backend is working - 1 files in state:

 /notes/todo.txt
----------------------------------------
{'content': '1. Record Video\n2. Edit Video\n3. Upload Video', 'encoding': 'utf-8', 'created_at': '2026-11-08T06:37:22.197510+00:00', 'modified_at': '2026-11-08T06:37:22.197510+00:00'}


# FileSystemBackend (Local Disk)

In [9]:
from deepagents.backends import FilesystemBackend

ROOT_DIR = "."

agent = create_deep_agent(
    model=llm,
    backend=FilesystemBackend(root_dir=ROOT_DIR, virtual_mode=True)
)

print(f"Agent Created with FileSystemBackend(root_dir={ROOT_DIR!r}).")
print("Files written by the agent will apear on your ACTUAL DISK")

Agent Created with FileSystemBackend(root_dir='.').
Files written by the agent will apear on your ACTUAL DISK


In [10]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content: \n"
            "1. Record Video\n2. Edit Video\n3. Upload Video\n"
            "Then tell me you've done it"
        )
    }]
})

print("\n------- Agent Reply -----------------------------------------------------------")
print(result["messages"][-1].content)


------- Agent Reply -----------------------------------------------------------
I've created the file `/notes/todo.txt` with the requested content.


In [11]:
disk_path = Path(ROOT_DIR) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"FileSystemBackend is working - file exists on disk:")
    print(f"{disk_path.resolve()}\n{'-'*40}")
    print(disk_path.read_text())
else:
    print(f"Expected file not found at {disk_path.resolve()}")
    print("The agent may not have called the write tool, or the path mapping differs")

FileSystemBackend is working - file exists on disk:
D:\AI Engineering\Loop Engineering\notes\todo.txt
----------------------------------------
1. Record Video
2. Edit Video
3. Upload Video


In [12]:
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Read a file at /requirements/REQ001.txt"
            "Back to me verbatim"
        )
    }]
})

print("\n------- Agent Reply -----------------------------------------------------------")
print(result["messages"][-1].content)


------- Agent Reply -----------------------------------------------------------
REQUIREMENT: User Profile File Upload
Users shall be able to upload files to their profile.
Uploaded files should be stored securely.
The system should support common file types.


# StoreBackend

Creates a deep agent backed by a langgraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on different thread - something StateBackend can't do

In [13]:
from deepagents.backends import StoreBackend
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()

agent = create_deep_agent(
    model=llm,
    backend=StoreBackend(
        namespace=lambda rt: ("demo-user", ),
    ),
    store=store
)

print("Agent with StoreBackend is created now")

Agent with StoreBackend is created now


In [14]:
import os
import uuid

thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file notes/todo.txt with exactly this content:\n"
                "1. Read 10 pages of Deep work book\n2. Create 4 reels today.\n3. make sure your take all your medicines today"
                "Then tell me you've done it"
            )
        }]
    },
    config=thread_1
)

print("\n-----------Agent Reply (Thread 1)--------------------------------------------------------------------------------------------------")
print(result["messages"][-1].content)


-----------Agent Reply (Thread 1)--------------------------------------------------------------------------------------------------
I've created the file `notes/todo.txt` with your specified content.


In [15]:
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}   
followup = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": "Read /notes/todo.txt back to me verbatim"
        }]
    },
    config=thread_2
)

print("\n--------READ BACK on a different thread-----------------------------------------------------------------------------------------------------")
print(followup["messages"][-1].content)


--------READ BACK on a different thread-----------------------------------------------------------------------------------------------------
[{'type': 'text', 'text': '1. Read 10 pages of Deep work book\n2. Create 4 reels today.\n3. make sure your take all your medicines today', 'extras': {'signature': 'CosBAWkUfRPxeTzMC9oHLiJv0DGm43EoN23Xkh2UJNQDu1i1vzZ9ldkllHD/UT5n/QCDnuPeMtiXax06OxY6V/wShMtbWfpWrEm83SFEMy8lCj5Exw9gvEQ7GWZaOY5t4oDNnY8WYtoytcRxqmPy2+fU0SqpdhBO6agOyv0D7qMiQkvgG4UFYpoYCIlzTg=='}}]
